In [ ]:
from pathlib import Path
import pandas as pd

FB = Path.home() / "datasets" / "financebench" / "data"
q = pd.read_json(FB / "financebench_open_source.jsonl", lines=True)
docs = pd.read_json(FB / "financebench_document_information.jsonl", lines=True)

print("Questions:", q.shape)
print("Documents:", docs.shape)
print("\nQuestion columns:", list(q.columns))
print("\nDocument columns:", list(docs.columns))

In [ ]:
row = q.iloc[0]   # the first question

for col in q.columns:
    if col != "evidence":
        print(f"{col}: {row[col]}")

print("\n--- evidence (where the answer is) ---")
for ev in row["evidence"]:
    for key, value in ev.items():
        text = str(value)
        print(f"{key}: {text[:300]}{'...' if len(text) > 300 else ''}")
    print()

In [ ]:
print(docs["doc_type"].value_counts())

In [ ]:
df = q.merge(docs, on="doc_name", how="left", suffixes=("", "_doc"))
tenk = df[df["doc_type"].str.lower().str.contains("10k")]

per_company = (
    tenk.groupby("company")
    .agg(
        n_reports=("doc_name", "nunique"),
        n_questions=("doc_name", "size"),
        years=("doc_period", lambda s: sorted(s.unique())),
    )
    .sort_values("n_questions", ascending=False)
)
per_company

In [ ]:
CHOSEN = {
    # company name exactly as in FinanceBench : stock ticker
    "AMD": "AMD",
    "Boeing": "BA",
    "American Express": "AXP",
    "PepsiCo": "PEP",
    "3M": "MMM",
    "Verizon": "VZ",
}

# The notebook runs inside notebooks/, so the project root is one level up
ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), f"Unexpected folder: {ROOT}"

# Find the sector column (its real name has a typo in FinanceBench)
sector_col = [c for c in df.columns if "sector" in c][0]

# Keep only 10-K reports that actually have questions, for the chosen companies
chosen_q = tenk[tenk["company"].isin(CHOSEN)]

manifest = (
    chosen_q[["doc_name", "company", "doc_period", sector_col]]
    .drop_duplicates("doc_name")
    .rename(columns={"doc_period": "fiscal_year", sector_col: "sector"})
    .assign(
        ticker=lambda d: d["company"].map(CHOSEN),
        doc_type="10-K",
    )
    [["doc_name", "company", "ticker", "fiscal_year", "doc_type", "sector"]]
    .sort_values(["company", "fiscal_year"])
    .reset_index(drop=True)
)

out_path = ROOT / "data" / "manifest.csv"
manifest.to_csv(out_path, index=False)

print(f"Reports:   {len(manifest)}")
print(f"Questions: {len(chosen_q)}")
print(f"Saved to:  {out_path}")
manifest

In [ ]:
picked = tenk[tenk["company"].isin(CHOSEN)].sample(8, random_state=42)

for i, (_, r) in enumerate(picked.iterrows(), 1):
    pages = sorted({ev["evidence_page_num"] + 1 for ev in r["evidence"]})  # +1 = human page numbers
    print(f"Q{i}: {r['question']}")
    print(f"  Expected: {r['answer']}")
    print(f"  Source:   {r['doc_name']}, page(s) {pages}")
    print()